# SVN-FCA-IC (second version) on the WDBC data: all results of the paper

**How to run:** Colab menu *Runtime, then Run all*. No upload is needed; the data come from scikit-learn. Expected run time: about 8 to 15 minutes.

**What changes compared with the first version:** the fixed purity threshold of 0.85 is replaced by a class-prior-relative filter, the class evidence uses the k strongest rules, and the filter parameters (delta, k) are chosen by inner cross-validation on the training data only. The first version is kept as a comparison ("Fixed 0.85 purity (first version)").

**What to send back:** `svnfca_v2_wdbc_results.zip` (downloaded automatically at the end). If the download is blocked, use the folder icon on the left.

In [ ]:
%pip -q install wittgenstein
import os, sys, json, time, platform, warnings
import numpy as np, pandas as pd, scipy, sklearn, matplotlib
warnings.filterwarnings("ignore")
os.makedirs("results", exist_ok=True)
env = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__,
           scikit_learn=sklearn.__version__, matplotlib=matplotlib.__version__)
try:
    import wittgenstein; env["wittgenstein"] = getattr(wittgenstein, "__version__", "installed")
except Exception as e:
    env["wittgenstein"] = str(e)
json.dump(env, open("results/environment.json", "w"), indent=1)
print(env)

## Model and evaluation code (written to three small files)

In [ ]:
%%writefile svnfca.py
"""Clean re-implementation of the SVN-FCA-IC pipeline found in the ChatGPT notebook.
All preprocessing parameters are learned on training data only."""
import numpy as np, pandas as pd
from sklearn.datasets import load_breast_cancer

CATS = ["low", "medium", "high"]


def load():
    d = load_breast_cancer()
    X = pd.DataFrame(d.data, columns=d.feature_names)
    y = pd.Series(np.where(d.target == 0, "M", "B"), name="diagnosis")
    return X, y


# ---------- crisp (tertile) context ----------
def fit_tertile_edges(Xtr):
    return np.quantile(Xtr, [1 / 3, 2 / 3], axis=0).T  # (p,2)


def crisp_context(X, edges):
    n, p = X.shape
    C = np.zeros((n, 3 * p), bool)
    for j in range(p):
        q1, q2 = edges[j]
        cat = np.where(X[:, j] <= q1, 0, np.where(X[:, j] <= q2, 1, 2))
        C[np.arange(n), 3 * j + cat] = True
    return C


# ---------- neutrosophic (SVNS) layer ----------
def fit_knots(Xtr):
    return np.quantile(Xtr, [0.2, 0.5, 0.8], axis=0).T  # (p,3)


def memberships(X, knots):
    n, p = X.shape
    T = np.zeros((n, p, 3))
    for j in range(p):
        a, b, c = knots[j]
        x = X[:, j]
        T[:, j, 0] = np.clip((b - x) / (b - a), 0, 1)
        left = np.clip((x - a) / (b - a), 0, 1)
        right = np.clip((c - x) / (c - b), 0, 1)
        T[:, j, 1] = np.minimum(left, right)
        T[:, j, 2] = np.clip((x - b) / (c - b), 0, 1)
    return T


def indeterminacy(T, kind="max"):
    if kind == "max":
        return 1.0 - T.max(axis=2)
    if kind == "entropy":
        with np.errstate(divide="ignore", invalid="ignore"):
            h = -np.where(T > 0, T * np.log(T), 0.0).sum(axis=2)
        return h / np.log(3)
    if kind == "none":
        return np.zeros(T.shape[:2])
    raise ValueError(kind)


def evidence_matrix(X, knots, ikind="max"):
    T = memberships(X, knots)
    I = indeterminacy(T, ikind)
    E = T * (1.0 - I[:, :, None])
    n, p, _ = E.shape
    return E.reshape(n, 3 * p), T, I


# ---------- rule mining (Apriori-style, identical rule set to brute force) ----------
def mine_rules(C, y, min_extent=10, min_purity=0.85, max_len=5, closed_only=False):
    n, m = C.shape
    yM = (np.asarray(y) == "M")
    feat = np.arange(m) // 3
    out = []
    level = [((j,), C[:, j]) for j in range(m) if C[:, j].sum() >= min_extent]
    k = 1
    while level and k <= max_len:
        for items, ext in level:
            size = int(ext.sum())
            nM = int((ext & yM).sum())
            nB = size - nM
            maj = "M" if nM > nB else "B"
            pur = max(nM, nB) / size
            if pur >= min_purity:
                closed = int(C[ext].all(axis=0).sum()) == len(items)
                if closed or not closed_only:
                    out.append(dict(items=items, cls=maj, extent=size,
                                    support=size / n, purity=pur, closed=closed,
                                    ext=ext.copy()))
        if k == max_len:
            break
        nxt = []
        for items, ext in level:
            last = items[-1]
            for j in range(last + 1, m):
                if feat[j] <= feat[last]:
                    continue
                e2 = ext & C[:, j]
                if e2.sum() >= min_extent:
                    nxt.append((items + (j,), e2))
        level = nxt
        k += 1
    return out


class RuleBase:
    def __init__(self, rules, m):
        self.rules = rules
        self.m = m
        r = len(rules)
        self.R = np.zeros((r, m))
        for i, ru in enumerate(rules):
            self.R[i, list(ru["items"])] = 1
        self.len = self.R.sum(1)
        self.isM = np.array([ru["cls"] == "M" for ru in rules], bool)
        self.pur = np.array([ru["purity"] for ru in rules])

    def __len__(self):
        return len(self.rules)

    def soft_scores(self, E):
        """geometric mean of per-condition evidence; 0 if any condition has no evidence."""
        pos = E > 1e-12
        LE = np.where(pos, np.log(np.where(pos, E, 1.0)), 0.0)
        Z = (~pos).astype(float)
        nz = Z @ self.R.T
        sl = LE @ self.R.T
        return np.where(nz == 0, np.exp(sl / self.len), 0.0)

    def crisp_scores(self, C):
        return ((C.astype(float) @ self.R.T) == self.len).astype(float)

    def aggregate(self, S):
        W = S * self.pur
        b = W[:, ~self.isM].sum(1)
        mm = W[:, self.isM].sum(1)
        return b, mm


def decide(b, m):
    tot = b + m
    pred = np.where(b > m, "B", "M").astype(object)
    pred[(tot == 0) | np.isclose(b, m)] = "abstain"
    margin = np.where(tot > 0, (b - m) / np.where(tot > 0, tot, 1), 0.0)
    return pred, margin


def run_svn(rb, Xte, Cte, knots, mode="svn", ikind="max"):
    if mode == "crisp":
        S = rb.crisp_scores(Cte)
    else:
        E, _, _ = evidence_matrix(Xte, knots, "none" if mode == "T" else ikind)
        S = rb.soft_scores(E)
    b, m = rb.aggregate(S)
    pred, margin = decide(b, m)
    return pred, margin, b, m


def metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred, dtype=object)
    acc_mask = y_pred != "abstain"
    cov = acc_mask.mean()
    yt, yp = y_true[acc_mask], y_pred[acc_mask]
    if len(yt) == 0:
        return dict(coverage=cov)
    tp = int(((yt == "M") & (yp == "M")).sum())
    fn = int(((yt == "M") & (yp == "B")).sum())
    fp = int(((yt == "B") & (yp == "M")).sum())
    tn = int(((yt == "B") & (yp == "B")).sum())
    rec = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    prec = tp / (tp + fp) if tp + fp else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return dict(coverage=cov, accuracy=(tp + tn) / len(yt), balanced_accuracy=(rec + spec) / 2,
                precision_M=prec, recall_M=rec, specificity=spec, f1_M=f1,
                tp=tp, fn=fn, fp=fp, tn=tn, n_accepted=len(yt))


In [ ]:
%%writefile svnfca2.py
"""SVN-FCA-IC, version 2: class-prior-aware rule filter, top-k class evidence and
tertile-consistent neutrosophic memberships. Every learned quantity (cut points, knots, rules,
filter parameters) is estimated from the training data only; filter parameters are chosen by
inner cross-validation on the training data."""
import math
import numpy as np
from sklearn.model_selection import StratifiedKFold
from svnfca import crisp_context, fit_tertile_edges

EPS = 1e-12


# ---------- neutrosophic memberships consistent with the tertile context ----------
def fit_knots_consistent(Xtr):
    """Knots (a, b, c) such that the crossing points of the memberships are the tertile cut points
    and the peak of 'medium' is the median: (a+b)/2 = q33, b = q50, (b+c)/2 = q67."""
    q = np.quantile(Xtr, [1 / 3, .5, 2 / 3], axis=0).T
    return np.stack([2 * q[:, 0] - q[:, 1], q[:, 1], 2 * q[:, 2] - q[:, 1]], axis=1)


def fit_knots_percentile(Xtr):
    return np.quantile(Xtr, [.2, .5, .8], axis=0).T


def memberships(X, knots):
    n, p = X.shape
    T = np.zeros((n, p, 3))
    for j in range(p):
        a, b, c = knots[j]
        wl, wr = max(b - a, EPS), max(c - b, EPS)
        x = X[:, j]
        T[:, j, 0] = np.clip((b - x) / wl, 0, 1)
        T[:, j, 1] = np.minimum(np.clip((x - a) / wl, 0, 1), np.clip((c - x) / wr, 0, 1))
        T[:, j, 2] = np.clip((x - b) / wr, 0, 1)
    return T


def evidence(X, knots, kind="max"):
    """kind: 'max' (I = 1 - max T), 'none' (I = 0, truth only) or 'entropy' (normalised entropy of the memberships)."""
    T = memberships(X, knots)
    if kind == "max":
        I = 1.0 - T.max(axis=2)
    elif kind == "entropy":
        with np.errstate(divide="ignore", invalid="ignore"):
            I = -np.where(T > 0, T * np.log(T), 0.0).sum(axis=2) / np.log(3)
    else:
        I = np.zeros(T.shape[:2])
    E = T * (1.0 - I[:, :, None])
    return E.reshape(len(X), -1)


# ---------- candidate antecedents ----------
def mine_candidates(C, min_extent, max_len=5):
    """All attribute sets with at most one category per feature, at least min_extent objects, length <= max_len."""
    n, m = C.shape
    feat = np.arange(m) // 3
    out, level, k = [], [((j,), C[:, j]) for j in range(m) if C[:, j].sum() >= min_extent], 1
    while level and k <= max_len:
        out += level
        if k == max_len:
            break
        nxt = []
        for items, ext in level:
            last = items[-1]
            for j in range(last + 1, m):
                if feat[j] <= feat[last]:
                    continue
                e2 = ext & C[:, j]
                if e2.sum() >= min_extent:
                    nxt.append((items + (j,), e2))
        level, k = nxt, k + 1
    return out


class Candidates:
    def __init__(self, C, yM, min_extent, max_len=5):
        cand = mine_candidates(C, min_extent, max_len)
        self.m = C.shape[1]
        self.items = [c[0] for c in cand]
        self.R = np.zeros((len(cand), self.m))
        for i, it in enumerate(self.items):
            self.R[i, list(it)] = 1
        self.len = self.R.sum(1)
        ext = np.array([c[1] for c in cand]) if cand else np.zeros((0, len(C)), bool)
        self.size = ext.sum(1).astype(float)
        self.nM = (ext & yM[None, :]).sum(1).astype(float)
        self.piM = float(yM.mean())
        self._C, self._ext, self._closed = C, ext, None

    @property
    def closed(self):
        if self._closed is None:
            self._closed = np.array([int(self._C[e].all(axis=0).sum()) == len(it) for it, e in zip(self.items, self._ext)], bool) if len(self.items) else np.zeros(0, bool)
        return self._closed

    def select(self, delta, relative=True):
        """Return (mask, isM, weight). Laplace-smoothed class purity; relative = excess over the class prior."""
        pM = (self.nM + 1) / (self.size + 2)
        pB = 1 - pM
        if relative:
            rM = (pM - self.piM) / max(1 - self.piM, EPS)
            rB = (pB - (1 - self.piM)) / max(self.piM, EPS)
        else:
            rM, rB = pM, pB
        isM = rM >= rB
        q = np.where(isM, rM, rB)
        return q >= delta, isM, q

    def soft(self, E):
        pos = E > 1e-12
        LE = np.where(pos, np.log(np.where(pos, E, 1.0)), 0.0)
        nz = (~pos).astype(float) @ self.R.T
        return np.where(nz == 0, np.exp(LE @ self.R.T / self.len), 0.0)

    def crisp(self, C):
        return ((C.astype(float) @ self.R.T) == self.len).astype(float)


def aggregate(S, mask, isM, w, k, agg):
    """Class evidence. 'topk': mean of the k strongest weighted rule scores; 'sum': sum of all."""
    W = S * w[None, :]
    out, used = [], []
    for cls_mask in (mask & ~isM, mask & isM):
        Wc = W[:, cls_mask]
        if Wc.shape[1] == 0:
            out.append(np.zeros(len(S))); used.append(np.zeros(len(S)))
        elif agg == "sum":
            out.append(Wc.sum(1)); used.append((Wc > 0).sum(1))
        else:
            kk = min(k, Wc.shape[1])
            top = -np.sort(-Wc, axis=1)[:, :kk]
            out.append(top.sum(1) / k); used.append(np.minimum((Wc > 0).sum(1), k))
    return out[0], out[1], used[0], used[1]  # evidence B, evidence M, rules used B, rules used M


def decide(b, m):
    tot = b + m
    pred = np.where(b > m, "B", "M").astype(object)
    pred[(tot == 0) | np.isclose(b, m)] = "abstain"
    margin = np.where(tot > 0, (b - m) / np.where(tot > 0, tot, 1), 0.0)
    return pred, margin


def bal_acc_abstain_wrong(y, pred):
    yM = np.asarray(y) == "M"
    rM = ((pred == "M") & yM).sum() / max(yM.sum(), 1)
    rB = ((pred == "B") & ~yM).sum() / max((~yM).sum(), 1)
    return (rM + rB) / 2


def min_extent_for(n, frac=0.02):
    return max(5, math.ceil(frac * n))


DELTAS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
KS = [1, 3, 5, 10]
THETAS = [0.60, 0.70, 0.80, 0.85, 0.90, 0.95]


def select_params(A, ya, relative=True, agg="topk", n_inner=5, seed=0, knots="consistent", min_frac=0.02, max_len=5):
    grid_d = DELTAS if relative else THETAS
    grid_k = KS if agg == "topk" else [1]
    sc = np.zeros((len(grid_d), len(grid_k)))
    fk = fit_knots_consistent if knots == "consistent" else fit_knots_percentile
    for tr, va in StratifiedKFold(n_inner, shuffle=True, random_state=seed).split(A, ya):
        e, kn = fit_tertile_edges(A[tr]), fk(A[tr])
        C = crisp_context(A[tr], e)
        cd = Candidates(C, ya[tr] == "M", min_extent_for(len(tr), min_frac), max_len)
        if len(cd.items) == 0:
            continue
        S = cd.soft(evidence(A[va], kn, "max"))
        for di, d in enumerate(grid_d):
            mask, isM, w = cd.select(d, relative)
            for ki, k in enumerate(grid_k):
                b, m, _, _ = aggregate(S, mask, isM, w, k, agg)
                pred, _ = decide(b, m)
                sc[di, ki] += bal_acc_abstain_wrong(ya[va], pred) / n_inner
    best = sc.max()
    ok = np.argwhere(sc >= best - 0.005)
    di, ki = max(ok.tolist(), key=lambda t: (t[0], -t[1]))  # parsimony: stricter filter, then smaller k
    return grid_d[di], grid_k[ki], sc


class SVN2:
    def __init__(self, relative=True, agg="topk", knots="consistent", closed_only=False, min_frac=0.02, max_len=5):
        self.relative, self.agg, self.knots, self.closed_only, self.min_frac, self.max_len = relative, agg, knots, closed_only, min_frac, max_len

    def fit(self, A, ya, delta=None, k=None, seed=0):
        if delta is None:
            delta, k, self.grid = select_params(A, ya, self.relative, self.agg, seed=seed, knots=self.knots, min_frac=self.min_frac, max_len=self.max_len)
        self.delta, self.k = delta, k
        self.edges = fit_tertile_edges(A)
        self.kn = (fit_knots_consistent if self.knots == "consistent" else fit_knots_percentile)(A)
        C = crisp_context(A, self.edges)
        self.cd = Candidates(C, ya == "M", min_extent_for(len(A), self.min_frac), self.max_len)
        mask, self.isM, self.w = self.cd.select(delta, self.relative)
        if self.closed_only:
            mask &= self.cd.closed
        self.mask = mask
        self.n_rules = int(mask.sum())
        self.n_rules_M = int((mask & self.isM).sum())
        return self

    def evidence_scores(self, B, mode="svn"):
        if len(self.cd.items) == 0:
            z = np.zeros(len(B)); return z, z, z, z
        if mode == "crisp":
            S = self.cd.crisp(crisp_context(B, self.edges))
        else:
            S = self.cd.soft(evidence(B, self.kn, {"svn": "max", "T": "none", "entropy": "entropy"}[mode]))
        return aggregate(S, self.mask, self.isM, self.w, self.k, self.agg)

    def predict(self, B, mode="svn"):
        b, m, ub, um = self.evidence_scores(B, mode)
        self.last_used = np.where(b > m, ub, um)
        pred, margin = decide(b, m)
        tot = b + m
        score = np.where(tot > 0, (m - b) / np.where(tot > 0, tot, 1), 0.0)
        return pred, margin, score


In [ ]:
%%writefile svn_eval.py
"""Evaluation helpers shared by both notebooks: baselines, one cross-validation fold, statistics."""
import warnings
import numpy as np, pandas as pd
from scipy import stats
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import wittgenstein as lw
from svnfca import fit_tertile_edges, fit_knots, crisp_context, mine_rules, RuleBase, run_svn, metrics
from svnfca2 import SVN2, bal_acc_abstain_wrong

warnings.filterwarnings("ignore")
MAIN = "SVN-FCA-IC"
_STORE = None


def top_k(A, ya, k=5):
    """Indices of the k features with the largest ANOVA F value (training fold only)."""
    if A.shape[1] <= k:
        return np.arange(A.shape[1])
    F, _ = f_classif(A, ya == "M")
    return np.sort(np.argsort(-np.nan_to_num(F))[:k])


def base_models(n_trees=300):
    sc = lambda m: make_pipeline(StandardScaler(), m)
    return {
        "LR": sc(LogisticRegression(max_iter=5000, random_state=42)),
        "DT (depth 5)": DecisionTreeClassifier(max_depth=5, random_state=42),
        "RF": RandomForestClassifier(n_estimators=n_trees, random_state=42),
        "SVM-RBF": sc(SVC(probability=True, random_state=42)),
        "kNN": sc(KNeighborsClassifier(5)),
        "GNB": GaussianNB(),
        "LR (balanced)": sc(LogisticRegression(max_iter=5000, random_state=42, class_weight="balanced")),
        "DT (depth 5, balanced)": DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced"),
        "RF (balanced)": RandomForestClassifier(n_estimators=n_trees, random_state=42, class_weight="balanced"),
        "SVM-RBF (balanced)": sc(SVC(probability=True, random_state=42, class_weight="balanced")),
    }


def ripper(A, ya, B):
    cn = [f"f{i}" for i in range(A.shape[1])]
    dfa = pd.DataFrame(A, columns=cn); dfa["y"] = (ya == "M")
    r = lw.RIPPER(random_state=0)
    r.fit(dfa, class_feat="y", pos_class=True)
    dfb = pd.DataFrame(B, columns=cn)
    pred = np.where(np.array(r.predict(dfb)), "M", "B").astype(object)
    return pred, np.asarray(r.predict_proba(dfb))[:, 1], len(r.ruleset_)


def _row(yb, pred, score, name, fi, rep, n_rules, feats, extra=None):
    d = metrics(yb, pred)
    d.update(model=name, fold=fi, rep=rep, n_rules=n_rules, feats=feats,
             auc=roc_auc_score(yb == "M", score), bal_all=bal_acc_abstain_wrong(yb, np.asarray(pred, dtype=object)))
    if extra:
        d.update(extra)
    if _STORE is not None:
        _STORE[(name, feats)] = (np.asarray(pred, dtype=object), np.asarray(score, float))
    return d


def one_fold(Xa, yv, tr, va, fi, rep, fixed_cols=None, ablations=False, refs_all=True, v1=True, n_trees=300):
    rows = []
    A_all, B_all, ya, yb = Xa[tr], Xa[va], yv[tr], yv[va]
    cols = np.asarray(fixed_cols) if fixed_cols is not None else top_k(A_all, ya)
    A, B = A_all[:, cols], B_all[:, cols]
    ft = str(len(cols))
    full = SVN2().fit(A, ya, seed=fi)
    p, mg, s = full.predict(B, "svn")
    rows.append(_row(yb, p, s, MAIN, fi, rep, full.n_rules, ft,
                     dict(delta=full.delta, kk=full.k, rules_used=float(np.mean(full.last_used)), n_rules_M=full.n_rules_M, margin_mean=float(np.mean(np.abs(mg))))))
    oof = pd.DataFrame(dict(fold=fi, rep=rep, y=yb, pred=p, margin=mg, score=s))
    cl = SVN2(closed_only=True).fit(A, ya, full.delta, full.k)
    p, _, s = cl.predict(B); rows.append(_row(yb, p, s, "Closed-intent rules", fi, rep, cl.n_rules, ft))
    if ablations:
        for name, mode in [("Crisp matching", "crisp"), ("Truth only (I = 0)", "T"), ("Entropy indeterminacy", "entropy")]:
            p, _, s = full.predict(B, mode); rows.append(_row(yb, p, s, name, fi, rep, full.n_rules, ft))
        for name, mdl in [("Absolute purity filter", SVN2(relative=False)), ("Sum of rule evidence", SVN2(agg="sum")), ("Percentile knots", SVN2(knots="percentile"))]:
            mdl.fit(A, ya, seed=fi); p, _, s = mdl.predict(B)
            rows.append(_row(yb, p, s, name, fi, rep, mdl.n_rules, ft, dict(delta=mdl.delta, kk=mdl.k)))
    if v1:  # original model: fixed purity 0.85, summed evidence, percentile knots
        e, k = fit_tertile_edges(A), fit_knots(A)
        C, Cb = crisp_context(A, e), crisp_context(B, e)
        rb = RuleBase(mine_rules(C, ya, 10, .85, 5), C.shape[1])
        p, mg1, b, m = run_svn(rb, B, Cb, k, "svn")
        tot = b + m
        rows.append(_row(yb, p, np.where(tot > 0, (m - b) / np.where(tot > 0, tot, 1), 0.0), "Fixed 0.85 purity (first version)", fi, rep, len(rb), ft))
    bm = base_models(n_trees)
    for name, mdl in bm.items():
        m_ = clone(mdl).fit(A, ya)
        rows.append(_row(yb, m_.predict(B), m_.predict_proba(B)[:, list(m_.classes_).index("M")], name, fi, rep, 0, ft))
    p, s, nr = ripper(A, ya, B); rows.append(_row(yb, p, s, "RIPPER", fi, rep, nr, ft))
    if refs_all and A_all.shape[1] > len(cols):
        for name in ["LR", "RF", "SVM-RBF", "LR (balanced)", "RF (balanced)", "SVM-RBF (balanced)"]:
            m_ = clone(bm[name]).fit(A_all, ya)
            rows.append(_row(yb, m_.predict(B_all), m_.predict_proba(B_all)[:, list(m_.classes_).index("M")], name, fi, rep, 0, "all"))
    return rows, oof


def run_cv(Xa, yv, n_repeats=10, seed=2024, **kw):
    rows, oofs = [], []
    for fi, (tr, va) in enumerate(RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats, random_state=seed).split(Xa, yv)):
        r, o = one_fold(Xa, yv, tr, va, fi, fi // 5, **kw)
        rows += r; oofs.append(o)
    return pd.DataFrame(rows), pd.concat(oofs, ignore_index=True)


def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); out = np.empty_like(p); run = 0.0
    for i, j in enumerate(o):
        run = max(run, (len(p) - i) * p[j]); out[j] = min(1.0, run)
    return out


def corrected_t(d, k_ratio=0.25):
    """Corrected resampled t-test (Nadeau & Bengio, 2003) on paired fold differences d."""
    d = np.asarray(d, float); k = len(d); v = d.var(ddof=1)
    if v <= 0:
        return 0.0, 1.0
    t = d.mean() / np.sqrt((1 / k + k_ratio) * v)
    return t, 2 * stats.t.sf(abs(t), k - 1)


def summarise(cv, ref=MAIN):
    cols = ["accuracy", "balanced_accuracy", "precision_M", "recall_M", "f1_M", "auc", "bal_all"]
    g = cv.groupby(["feats", "model"])
    out = g[cols + ["n_rules", "coverage"]].mean().join(g[["accuracy", "balanced_accuracy", "auc"]].std().add_suffix("_std")).reset_index()
    r = cv[cv.model == ref].sort_values("fold")
    for metric in ["accuracy", "balanced_accuracy", "auc"]:
        ps = []
        for (f, mname), gg in cv.groupby(["feats", "model"]):
            if mname == ref and f == r.feats.iloc[0]:
                ps.append(np.nan); continue
            d = r[metric].values - gg.sort_values("fold")[metric].values
            ps.append(corrected_t(d)[1])
        out["p_" + metric] = ps
        mk = out["p_" + metric].notna().values
        out.loc[mk, "p_" + metric + "_holm"] = holm(out.loc[mk, "p_" + metric].values)
    return out


## Experiment 1: repeated stratified 5-fold cross-validation (10 repeats), five mean features

In [ ]:
import svn_eval
from svn_eval import *
from svnfca import load, CATS
X, y = load(); Xall, yv = X.values, y.values; F5 = list(X.columns[:5])
print("WDBC:", Xall.shape, pd.Series(yv).value_counts().to_dict())
t0 = time.time()
cv, oof = run_cv(Xall, yv, n_repeats=10, seed=2024, fixed_cols=[0, 1, 2, 3, 4], ablations=True, refs_all=True)
cv.to_csv("results/cv_folds.csv", index=False); oof.to_csv("results/cv_oof_main.csv", index=False)
cvs = summarise(cv); cvs.to_csv("results/cv_table.csv", index=False)
print("elapsed", round(time.time() - t0), "s")
show = ["feats", "model", "accuracy", "balanced_accuracy", "auc", "n_rules", "coverage", "p_balanced_accuracy", "p_balanced_accuracy_holm"]
print(cvs[show].round(4).to_string(index=False))
sel = cv[cv.model == MAIN]
print("\nSelected filter parameters over the 50 training folds:\n", sel[["delta", "kk", "rules_used", "n_rules"]].describe().round(3))
print(sel.delta.value_counts().sort_index().to_dict(), sel.kk.value_counts().sort_index().to_dict())
R = {"cv_selected": dict(delta=sel.delta.value_counts().to_dict(), k=sel.kk.value_counts().to_dict(), rules_used_mean=float(sel.rules_used.mean()))}

## Experiment 2: stratified 80/20 hold-out (seed 42)

In [ ]:
from sklearn.model_selection import train_test_split
from scipy.stats import binomtest
n = len(yv); tr, va = train_test_split(np.arange(n), test_size=.2, random_state=42, stratify=yv)
svn_eval._STORE = {}
rows, _ = one_fold(Xall, yv, tr, va, 0, 0, fixed_cols=[0, 1, 2, 3, 4], ablations=True, refs_all=True)
store = svn_eval._STORE; svn_eval._STORE = None
ho = pd.DataFrame(rows); ho.to_csv("results/holdout_table.csv", index=False)
yte = yv[va]; print("hold-out:", len(tr), "train /", len(va), "test; malignant in test:", int((yte == "M").sum()))
print(ho[["feats", "model", "accuracy", "balanced_accuracy", "precision_M", "recall_M", "auc", "tp", "fn", "fp", "tn", "coverage", "n_rules"]].round(4).to_string(index=False))
ok = {k: (v[0] == yte) for k, v in store.items()}      # abstention counts as an error
rng = np.random.default_rng(7); idx = rng.integers(0, len(yte), size=(5000, len(yte)))
mc, ci = {}, {}
for (name, ft), c in ok.items():
    ci[f"{name} ({ft})"] = [float(np.percentile(c[idx].mean(1), 2.5)), float(np.percentile(c[idx].mean(1), 97.5))]
    if (name, ft) != (MAIN, "5"):
        a, b = ok[(MAIN, "5")], c
        n10, n01 = int((a & ~b).sum()), int((~a & b).sum())
        mc[f"{name} ({ft})"] = dict(svn_only_correct=n10, other_only_correct=n01, p=float(binomtest(min(n10, n01), n10 + n01, .5).pvalue) if n10 + n01 else 1.0,
                                    diff=float(a.mean() - b.mean()))
pd.DataFrame(mc).T.round(4).to_csv("results/holdout_mcnemar.csv")
print(pd.DataFrame(mc).T.round(4).to_string()); print({k: [round(x, 3) for x in v] for k, v in ci.items() if k.startswith(MAIN)})
R["holdout_mcnemar"], R["holdout_boot_ci"] = mc, ci
R["train_test_sizes"] = [int(len(tr)), int(len(va))]

## Experiment 3: rule base, formal concepts and worked cases

In [ ]:
from svnfca import fit_tertile_edges, crisp_context
from svnfca2 import SVN2, Candidates, fit_knots_consistent, evidence, memberships, min_extent_for
A, B, ya, yb = Xall[tr][:, :5], Xall[va][:, :5], yv[tr], yv[va]
full = SVN2().fit(A, ya, seed=0)
cd = full.cd; names = [f"{F5[j // 3]} = {CATS[j % 3]}" for j in range(15)]
Cb = crisp_context(B, full.edges); crisp_hits = cd.crisp(Cb); ytM = yb == "M"
rows = []
for i in np.where(full.mask)[0]:
    cls = "M" if full.isM[i] else "B"; hit = crisp_hits[:, i] > 0; nt = int(hit.sum())
    pur = (cd.nM[i] if cls == "M" else cd.size[i] - cd.nM[i]) / cd.size[i]
    tp = float((ytM[hit] if cls == "M" else ~ytM[hit]).mean()) if nt else np.nan
    rows.append(dict(antecedent=" AND ".join(names[j] for j in cd.items[i]), cls=cls, length=len(cd.items[i]), extent=int(cd.size[i]),
                     support=cd.size[i] / len(A), purity=pur, weight=float(full.w[i]), closed=bool(cd.closed[i]), n_test=nt, test_prec=tp))
rdf = pd.DataFrame(rows); rdf.to_csv("results/rules_all.csv", index=False)
ext_key = [cd._ext[i].tobytes() for i in np.where(full.mask)[0]]
rdf["ext_key"] = ext_key
rep = rdf.sort_values(["weight", "extent", "length"], ascending=[False, False, True]).drop_duplicates("ext_key"); rep.drop(columns="ext_key").to_csv("results/rules_rep.csv", index=False)
print("selected delta, k:", full.delta, full.k, " candidates:", len(cd.items), " rules kept:", full.n_rules, "(M:", full.n_rules_M, ")  prior M:", round(cd.piM, 4))
print(rdf.groupby("cls")[["length", "extent", "purity", "weight"]].mean().round(3)); print("by length:", rdf.length.value_counts().sort_index().to_dict(), " closed:", int(rdf.closed.sum()), " distinct extents:", rdf.ext_key.nunique())
ev = rdf[rdf.n_test > 0]
R["rule_base"] = dict(delta=full.delta, k=full.k, prior_M=float(cd.piM), n_candidates=len(cd.items), n_rules=full.n_rules, n_rules_M=full.n_rules_M, n_closed=int(rdf.closed.sum()),
                      n_distinct_extents=int(rdf.ext_key.nunique()), min_extent=int(min_extent_for(len(A))), n_eval=len(ev), mean_train_purity=float(ev.purity.mean()),
                      mean_test_precision=float(ev.test_prec.mean()), weighted_test_precision=float((ev.test_prec * ev.n_test).sum() / ev.n_test.sum()),
                      n_rules_test_prec_below_train_purity_minus_0_1=int((ev.test_prec < ev.purity - 0.1).sum()))
print(R["rule_base"])
print("\nTop distinct-extent rules")
for c in ["B", "M"]: print(c); print(rep[rep.cls == c].head(5)[["antecedent", "length", "extent", "purity", "weight", "n_test", "test_prec"]].round(3).to_string(index=False))
# facts quoted in the paper: cut points, knots, formal concepts, correlations
edges_, knots_ = full.edges, full.kn
Cc = crisp_context(A, edges_); m_ = Cc.shape[1]; cols_ = [Cc[:, j] for j in range(m_)]; n_conc = 0
for mask in range(1 << m_):
    e = np.ones(len(Cc), bool)
    for j in range(m_):
        if mask >> j & 1: e &= cols_[j]
    clo = int(sum(1 << j for j in range(m_) if Cc[e].all(axis=0)[j])) if e.any() else (1 << m_) - 1
    n_conc += (clo == mask)
facts = dict(n_train=len(A), n_test=len(B), tertile_cuts={F5[j]: [round(float(v), 4) for v in edges_[j]] for j in range(5)},
             fuzzy_knots={F5[j]: [round(float(v), 4) for v in knots_[j]] for j in range(5)}, n_formal_concepts=int(n_conc),
             corr_train=pd.DataFrame(A, columns=F5).corr().round(3).to_dict())
json.dump(facts, open("results/facts.json", "w"), indent=1); print("formal concepts:", n_conc)
# two worked cases
S_ = cd.soft(evidence(B, full.kn, "max")); pred, mg, sc = full.predict(B)
Tm = memberships(B, full.kn); Ev = evidence(B, full.kn, "max")
def explain(i):
    wsc = np.where(full.mask, S_[i] * full.w, 0.0); top = np.argsort(-wsc)[:4]
    b, m, _, _ = full.evidence_scores(B[i:i + 1])
    return dict(index=int(va[i]), actual=str(yb[i]), pred=str(pred[i]), evB=float(b[0]), evM=float(m[0]), margin=float(mg[i]),
                values={F5[j]: float(B[i, j]) for j in range(5)}, T={F5[j]: [float(x) for x in Tm[i, j]] for j in range(5)},
                I={F5[j]: float(1 - Tm[i, j].max()) for j in range(5)}, crisp={F5[j]: CATS[int(np.where(Cb[i, 3 * j:3 * j + 3])[0][0])] for j in range(5)},
                top_rules=[dict(rule=rdf.antecedent.iloc[list(np.where(full.mask)[0]).index(k)], cls="M" if full.isM[k] else "B", weight=float(full.w[k]), score=float(S_[i, k])) for k in top if wsc[k] > 0],
                n_active_B=int(((S_[i] > 0) & full.mask & ~full.isM).sum()), n_active_M=int(((S_[i] > 0) & full.mask & full.isM).sum()))
fn = [i for i in range(len(yb)) if yb[i] == "M" and pred[i] != "M"]; okM = sorted([i for i in range(len(yb)) if yb[i] == "M" and pred[i] == "M"], key=lambda i: abs(mg[i]))
ex = {"correct_M": explain(okM[len(okM) // 2])}
if fn: ex["fn"] = explain(fn[int(np.argmin([abs(mg[i]) for i in fn]))])
ex["fn_all_margins"] = [float(mg[i]) for i in fn]; json.dump(ex, open("results/example.json", "w"), indent=1)
print("hold-out errors:", int((pred != yb).sum()), " false negatives:", len(fn)); print(json.dumps(ex, indent=1)[:2500])

## Experiment 4: one-at-a-time sensitivity (training part only)

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold
d0 = float(sel.delta.median()); k0 = int(sel.kk.median()); print("centre of the one-at-a-time analysis: delta =", d0, " k =", k0)
def cv_fixed(delta, k, min_frac=0.02, max_len=5):
    acc, bal, nr = [], [], []
    for a, b in RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=2024).split(A, ya):
        m = SVN2(min_frac=min_frac, max_len=max_len).fit(A[a], ya[a], delta, k)
        p = m.predict(A[b])[0]; acc.append(np.mean(p == ya[b])); bal.append(bal_acc_abstain_wrong(ya[b], p)); nr.append(m.n_rules)
    return dict(acc=float(np.mean(acc)), acc_sd=float(np.std(acc, ddof=1)), bal=float(np.mean(bal)), rules=float(np.mean(nr)))
sens = []
for d in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]: sens.append(dict(param="delta", value=d, **cv_fixed(d, k0)))
for k in [1, 2, 3, 5, 10, 20]: sens.append(dict(param="k", value=k, **cv_fixed(d0, k)))
for f in [0.01, 0.02, 0.05, 0.1]: sens.append(dict(param="min_extent_fraction", value=f, **cv_fixed(d0, k0, min_frac=f)))
for L in [1, 2, 3, 4, 5]: sens.append(dict(param="max_length", value=L, **cv_fixed(d0, k0, max_len=L)))
sdf = pd.DataFrame(sens); sdf.to_csv("results/sensitivity.csv", index=False); print(sdf.round(4).to_string(index=False)); R["sensitivity"] = sens; R["sensitivity_centre"] = dict(delta=d0, k=k0)

## Experiment 5: selective classification (abstention) on the cross-validated margins

In [ ]:
o = oof.copy(); rows = []
for tau in [0, .05, .1, .2, .3, .4, .5, .6, .7, .8]:
    acc_mask = (o.pred != "abstain") & (o.margin.abs() >= tau)
    a = o[acc_mask]; per_rep = o.assign(acc=acc_mask).groupby("rep")
    err = a.assign(e=(a.pred != a.y)).groupby("rep").e.sum().reindex(range(10), fill_value=0)
    fn_ = a[(a.y == "M") & (a.pred != "M")].groupby("rep").size().reindex(range(10), fill_value=0); fp_ = a[(a.y == "B") & (a.pred != "B")].groupby("rep").size().reindex(range(10), fill_value=0)
    rows.append(dict(threshold=tau, coverage=float(acc_mask.mean()), accuracy=float((a.pred == a.y).mean()), errors=float(err.mean()), fn=float(fn_.mean()), fp=float(fp_.mean())))
ab = pd.DataFrame(rows); ab.to_csv("results/abstention.csv", index=False); print(ab.round(4).to_string(index=False)); R["abstention_cv"] = rows

## Experiment 6: ten mean features (growth of the rule base)

In [ ]:
t0 = time.time()
cv10, _ = run_cv(Xall[:, :10], yv, n_repeats=2, seed=2024, fixed_cols=list(range(10)), ablations=False, refs_all=False, v1=False)
c10 = summarise(cv10); c10.to_csv("results/ten_features.csv", index=False)
print("elapsed", round(time.time() - t0), "s"); print(c10[["model", "accuracy", "balanced_accuracy", "auc", "n_rules", "coverage"]].round(4).to_string(index=False))
R["ten_features"] = c10[["model", "accuracy", "balanced_accuracy", "auc", "n_rules", "coverage"]].round(4).to_dict(orient="records")

## Figures 1 to 4

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "Helvetica", "DejaVu Sans"], "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 120})
BLUE = "black"; GREY = "#4D4D4D"
# Fig 1: framework (one colour, white background)
fig, ax = plt.subplots(figsize=(7.2, 3.4), facecolor="white"); ax.axis("off"); ax.set_xlim(0, 100); ax.set_ylim(0, 46)
def box(x, y_, w, h, txt): ax.add_patch(FancyBboxPatch((x, y_), w, h, boxstyle="round,pad=0.3,rounding_size=1.2", fc="white", ec=BLUE, lw=1.2)); ax.text(x + w / 2, y_ + h / 2, txt, ha="center", va="center", fontsize=8.5)
def arrow(x1, y1, x2, y2): ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color=BLUE, lw=1.1))
box(1, 31, 20, 11, "Training data\ntertile cut points,\nfuzzy knots"); box(26, 31, 22, 11, "Formal context\n(binary attributes)\ncandidate rules"); box(53, 31, 21, 11, "Class-relative\npurity filter\n(inner CV)"); box(79, 31, 20, 11, "Rule base\nweights = excess\npurity over prior")
box(1, 10, 22, 11, "New case"); box(28, 10, 22, 11, "Neutrosophic evidence\nT, I, F per attribute\ne = T(1 - I)"); box(55, 10, 20, 11, "Rule scores\n(geometric mean)\ntop-k per class"); box(80, 10, 19, 11, "Class evidence,\nmargin, decision\nor abstention")
for x1, x2 in [(21.3, 25.7), (48.3, 52.7), (74.3, 78.7)]: arrow(x1, 36.5, x2, 36.5)
for x1, x2 in [(23.3, 27.7), (50.3, 54.7), (75.3, 79.7)]: arrow(x1, 15.5, x2, 15.5)
arrow(37, 30.6, 39, 21.8); arrow(89, 30.6, 65, 21.8)
ax.text(1, 3, "Upper row: learning. Lower row: inference.", fontsize=8.5, color="black")
fig.savefig("results/fig1.png", dpi=300, bbox_inches="tight", facecolor="white"); plt.show()
# Fig 2: memberships of mean radius (training part)
j = 0; xs = np.linspace(A[:, j].min(), A[:, j].max(), 600); Tm_ = memberships(np.repeat(xs[:, None], 5, 1), full.kn)[:, 0, :]
fig, ax = plt.subplots(figsize=(6.4, 3.1), facecolor="white")
for k_, (lab, ls, col) in enumerate([("low", "-", "#2F6DB5"), ("medium", "--", "#D9822B"), ("high", "-.", "#3A9A5B")]): ax.plot(xs, Tm_[:, k_], color=col, ls=ls, lw=1.9, label=f"T ({lab})")
ax.plot(xs, 1 - Tm_.max(1), color="black", lw=1.4, ls=":", label="I = 1 - max T")
for q in full.edges[j]: ax.axvline(q, color="#BBBBBB", lw=1, ls="--")
ax.set_xlabel("Mean radius"); ax.set_ylabel("Degree"); ax.set_ylim(0, 1.08); ax.legend(frameon=False, fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(.5, -.22))
fig.savefig("results/fig2.png", dpi=300, bbox_inches="tight", facecolor="white"); plt.show()
# Fig 3: training purity versus hold-out precision
d = rdf[rdf.n_test >= 3]; rng_ = np.random.default_rng(0)
fig, axs = plt.subplots(1, 2, figsize=(7.4, 3.6), sharey=True, facecolor="white")
for ax, (cl, mk, col, t_) in zip(axs, [("B", "o", "#2F6DB5", "(a) Rules predicting benign"), ("M", "s", "#D9622B", "(b) Rules predicting malignant")]):
    s_ = d[d.cls == cl]
    ax.scatter(s_.purity + rng_.uniform(-.004, .004, len(s_)), s_.test_prec, s=14 + 1.2 * s_.n_test, marker=mk, facecolor=col, edgecolor="black", lw=.5, alpha=.55)
    ax.plot([.8, 1.02], [.8, 1.02], color="#666666", lw=1, ls="--")
    ax.set_xlim(.85, 1.015); ax.set_ylim(.74, 1.03); ax.set_xlabel("Training purity"); ax.set_title(t_, fontsize=9, loc="left")
    ax.text(.853, .755, f"{len(s_)} rules; marker size = matched cases", fontsize=7.5)
axs[0].set_ylabel("Precision on matched hold-out cases")
fig.savefig("results/fig3.png", dpi=300, bbox_inches="tight", facecolor="white"); plt.show()
# Fig 4: risk-coverage
fig, ax = plt.subplots(figsize=(5.6, 3.4), facecolor="white"); ax.plot(ab.coverage * 100, ab.accuracy * 100, color="#2F6DB5", lw=1.8, marker="o", ms=5)
for _, r_ in ab.iterrows():
    if r_.threshold in (0, .2, .5, .8): ax.annotate(f"\u03c4 = {r_.threshold:g}", (r_.coverage * 100, r_.accuracy * 100), textcoords="offset points", xytext=((8, 4) if r_.threshold == 0 else (6, -12)), fontsize=8)
ax.set_xlabel("Coverage (%)"); ax.set_ylabel("Accuracy on accepted cases (%)"); ax.invert_xaxis()
fig.savefig("results/fig4.png", dpi=300, bbox_inches="tight", facecolor="white"); plt.show()

## Summary and download

In [ ]:
json.dump(R, open("results/results.json", "w"), indent=1, default=float)
t = ho[(ho.model == MAIN)].iloc[0]
print("HOLD-OUT", MAIN, "accuracy", round(t.accuracy, 4), "TP/FN/FP/TN", int(t.tp), int(t.fn), int(t.fp), int(t.tn), "AUC", round(t.auc, 4), "rules", int(t.n_rules))
print("CV", cvs[(cvs.model == MAIN)][["accuracy", "balanced_accuracy", "auc", "n_rules"]].round(4).to_dict("records"))
import shutil
shutil.make_archive("svnfca_v2_wdbc_results", "zip", "results"); print("Created svnfca_v2_wdbc_results.zip with:", sorted(os.listdir("results")))
try:
    from google.colab import files
    files.download("svnfca_v2_wdbc_results.zip")
except Exception as e:
    print("Automatic download not available:", e, "\nUse the folder icon on the left, open the zip file, and download it.")